# 階段 1：資料取得與品質檢查

台股擇時策略研究專案的第一步。

**目的**：下載台灣加權指數（`^TWII`）日線資料，做完整品質檢查，產出一份可信任的基礎資料檔，供後續階段使用。

**本階段不計算任何技術指標、不做任何回測。**

**產出**

| 檔案 | 內容 |
|---|---|
| `data/twii_raw.csv` | 原始日線資料，未經任何清理 |
| `data/data_quality_report.md` | 人類可讀的品質檢查報告 |

**關於樣本起點**：資料從 **1999-01-01** 開始下載，比研究樣本起點（2000-01-01）早一年。
多抓的這一年是暖身資料，讓 2000 年第一個交易日就能算出完整的 MA50。
後續績效統計會從 2000-01-01 起算，但資料檔完整保留 1999 年的部分。

---
## 1. 環境與參數

所有參數與判定門檻集中在這一格，日後要修改只需要改這裡。

同時定義 `record()` 這個小工具：每個檢查項目跑完就呼叫一次，把「通過／異常」記錄下來，
最後區塊 9 直接用這些記錄產生報告，避免報告與實際檢查結果脫節。

In [1]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# ============ 參數 ============
TICKER   = "^TWII"
START    = "1999-01-01"
END      = "2026-01-21"
DATA_DIR = "data"

RAW_CSV   = os.path.join(DATA_DIR, "twii_raw.csv")
REPORT_MD = os.path.join(DATA_DIR, "data_quality_report.md")

# ============ 判定門檻 ============
TRADING_DAYS_MIN = 230      # 每年交易日數合理下限
TRADING_DAYS_MAX = 255      # 每年交易日數合理上限
GAP_CALENDAR_MAX = 5        # 資料空窗超過幾個日曆日就列出來確認
OPEN_EQ_PREV_CLOSE_MAX = 0.05   # Open == 前一日 Close 的比例上限（超過視為開盤價不可信）
TOP_N_EXTREME = 15          # 極端變動檢視筆數

os.makedirs(DATA_DIR, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (14, 6)

# ============ 檢查結果收集器 ============
CHECKS = []   # 每個檢查項目的通過/異常
TABLES = {}   # 要放進報告的明細表

def record(name, passed, detail=""):
    """記錄一個檢查項目的結果，並立刻印出結論。"""
    verdict = "通過" if passed else "異常"
    CHECKS.append({"檢查項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("環境就緒")
print(f"  標的 : {TICKER}")
print(f"  期間 : {START} ~ {END}")
print(f"  輸出 : {RAW_CSV}")

環境就緒
  標的 : ^TWII
  期間 : 1999-01-01 ~ 2026-01-21
  輸出 : data\twii_raw.csv


---
## 2. 下載資料

從 yfinance 下載 `^TWII` 日線，保留 Open / High / Low / Close / Volume 五個欄位。

幾個刻意的處理方式：

- `auto_adjust=False`：不要讓 yfinance 自動調整價格。加權指數不配息、不分割，我們要的就是原始報價。
- yfinance 新版回傳 MultiIndex 欄位（`(欄位, 代號)`），只**攤平欄位名稱**，不動任何數值。
- 下載後**立刻原樣存檔**，不做任何清理。所有品質問題留到後面的區塊檢查，由人決定怎麼處理。
- 如果回傳空資料或筆數明顯不足，直接 `raise` 中斷，**不會自動改用其他資料來源或代號**。

In [2]:
raw = yf.download(TICKER, start=START, end=END, auto_adjust=False, progress=False)

# yfinance 新版回傳 MultiIndex 欄位，僅攤平名稱，數值不動
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

# 下載失敗或明顯不完整就停下來，不自行更換資料來源
if raw.empty:
    raise RuntimeError("下載失敗：yfinance 回傳空資料。請停止並人工確認網路 / 代號 / yfinance 版本。")
if len(raw) < 6000:
    raise RuntimeError(f"資料明顯不完整：只有 {len(raw)} 筆，26 年日線應約 6,600 筆。請停止並人工確認。")

raw = raw[["Open", "High", "Low", "Close", "Volume"]]
raw.index = pd.to_datetime(raw.index).tz_localize(None)
raw.index.name = "Date"

raw.to_csv(RAW_CSV, date_format="%Y-%m-%d")
print(f"已存檔：{RAW_CSV}")
print(f"筆數：{len(raw):,}    期間：{raw.index.min():%Y-%m-%d} ~ {raw.index.max():%Y-%m-%d}")

已存檔：data\twii_raw.csv
筆數：6,632    期間：1999-01-05 ~ 2026-01-20


讀回剛才存的 CSV 來做後續檢查。

這一步不是多餘的：後續階段讀的是這個檔案，用同一份讀回的資料做檢查，
才能確保「檢查通過的東西」和「下游拿到的東西」是同一份，而不是記憶體裡的另一個版本。

In [3]:
df = pd.read_csv(RAW_CSV, index_col="Date", parse_dates=True)
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
1999-01-05,"6,210.41","6,310.41","6,111.64","6,152.43",0
1999-01-06,"6,082.02","6,280.93","5,988.06","6,199.91",0
1999-01-07,"6,280.38","6,409.55","6,181.62","6,404.31",0
1999-01-08,"6,371.34","6,492.87","6,371.34","6,421.75",0
1999-01-11,"6,472.02","6,492.90","6,392.49","6,406.99",0


---
## 3. 基本資訊

三件事：資料的規模與範圍、各欄位缺值狀況、數值分布概覽。

缺值只要有任何一筆就算異常 —— 指數日線資料不應該有 NaN。

In [4]:
overview = pd.DataFrame({
    "值": [
        f"{len(df):,}",
        f"{df.index.min():%Y-%m-%d}",
        f"{df.index.max():%Y-%m-%d}",
        ", ".join(df.columns),
        str(df.index.dtype),
    ]
}, index=["總筆數", "起始日期", "結束日期", "欄位", "索引型別"])
overview.index.name = "項目"
display(overview)

,值
項目,
總筆數,"6,632"
起始日期,1999-01-05
結束日期,2026-01-20
欄位,"Open, High, Low, Close, Volume"
索引型別,datetime64[us]


In [5]:
na = df.isna().sum().to_frame("缺值數")
na["缺值比例"] = (na["缺值數"] / len(df)).map("{:.4%}".format)
na.index.name = "欄位"
display(na)

TABLES["缺值統計"] = na
n_na = int(na["缺值數"].sum())
record("欄位缺值", n_na == 0,
       "所有欄位皆無缺值" if n_na == 0 else f"共 {n_na} 個缺值，明細見上表")

,缺值數,缺值比例
欄位,,
Open,0,0.0000%
High,0,0.0000%
Low,0,0.0000%
Close,0,0.0000%
Volume,0,0.0000%


===> [通過] 欄位缺值
      所有欄位皆無缺值


In [6]:
desc = df.describe().T
display(desc)
TABLES["數值概覽"] = desc.round(2)

,count,mean,std,min,25%,50%,75%,max
Open,"6,632.00","9,938.94","4,951.37","3,475.87","6,736.31","8,412.51","10,835.14","31,584.34"
High,"6,632.00","9,993.98","4,977.14","3,511.38","6,780.55","8,461.13","10,876.17","31,827.39"
Low,"6,632.00","9,875.89","4,931.63","3,411.68","6,674.95","8,365.10","10,799.22","31,340.10"
Close,"6,632.00","9,933.12","4,955.10","3,446.26","6,719.77","8,412.32","10,828.88","31,759.99"
Volume,"6,632.00","2,823,965.67","1,835,201.85",0.00,"1,853,675.00","2,652,700.00","3,827,750.00","14,999,500.00"


---
## 4. 每年交易日數

台股一年約 240–250 個交易日。落在 **230–255 之外**的年份要標記出來人工確認，
可能代表該年度資料有缺漏。

2026 年只有不到一個月的資料（下載期間到 2026-01-21 為止），屬正常，標為「部分年度」不做判定。

In [7]:
last_year = int(df.index.max().year)

yearly = df.groupby(df.index.year).size().to_frame("交易日數")
yearly.index.name = "年份"
yearly["首個交易日"] = df.groupby(df.index.year).apply(lambda g: g.index.min().strftime("%Y-%m-%d"))
yearly["末個交易日"] = df.groupby(df.index.year).apply(lambda g: g.index.max().strftime("%Y-%m-%d"))

def year_status(row):
    if row.name == last_year:
        return "部分年度（不判定）"
    return "正常" if TRADING_DAYS_MIN <= row["交易日數"] <= TRADING_DAYS_MAX else "★ 異常"

yearly["狀態"] = yearly.apply(year_status, axis=1)
display(yearly)

,交易日數,首個交易日,末個交易日,狀態
年份,,,,
1999,241,1999-01-05,1999-12-28,正常
2000,245,2000-01-04,2000-12-29,正常
2001,245,2001-01-02,2001-12-31,正常
2002,248,2002-01-02,2002-12-31,正常
2003,249,2003-01-02,2003-12-31,正常
2004,250,2004-01-02,2004-12-31,正常
2005,247,2005-01-03,2005-12-30,正常
2006,247,2006-01-02,2006-12-29,正常
2007,243,2007-01-02,2007-12-31,正常


In [8]:
abnormal_years = yearly[yearly["狀態"].str.contains("異常")]

TABLES["每年交易日數"] = yearly
record("每年交易日數",
       abnormal_years.empty,
       f"1999–{last_year - 1} 年交易日數皆落在 {TRADING_DAYS_MIN}–{TRADING_DAYS_MAX} 之間"
       if abnormal_years.empty
       else "以下年份落在合理區間外：" + "、".join(
           f"{y} 年 {int(r['交易日數'])} 天" for y, r in abnormal_years.iterrows()))

===> [通過] 每年交易日數
      1999–2025 年交易日數皆落在 230–255 之間


---
## 5. 缺漏與重複

**重複日期**：同一天出現兩筆資料會讓後續所有計算失真，只要有就是異常。

**資料空窗**：列出所有間隔超過 5 個日曆日的空窗。台股農曆年封關約 5–9 天屬正常，
其他連假（清明、端午、中秋加上補假）也會產生空窗。
這一項**不自動判定對錯**，而是把所有空窗列出來讓人確認 —— 因為「正常長假」和「資料漏抓」
在數字上長得一樣，只能靠人比對日期。

In [9]:
dups = df.index[df.index.duplicated(keep=False)]
if len(dups) > 0:
    display(df.loc[dups])

record("重複日期", len(dups) == 0,
       "無重複日期" if len(dups) == 0 else f"發現 {df.index.duplicated().sum()} 筆重複日期")

===> [通過] 重複日期
      無重複日期


In [10]:
d = df.index.to_series()
gap = pd.DataFrame({
    "空窗起（前一交易日）": d.shift(1),
    "空窗迄（下一交易日）": d,
})
gap = gap.dropna()
gap["日曆間隔天數"] = (gap["空窗迄（下一交易日）"] - gap["空窗起（前一交易日）"]).dt.days

# 扣除週末：計算中間夾了幾個沒有資料的平日
gap["缺漏平日數"] = [
    len(pd.bdate_range(a + pd.Timedelta(days=1), b - pd.Timedelta(days=1)))
    for a, b in zip(gap["空窗起（前一交易日）"], gap["空窗迄（下一交易日）"])
]

gaps = gap[gap["日曆間隔天數"] > GAP_CALENDAR_MAX].copy()
gaps["空窗起（前一交易日）"] = gaps["空窗起（前一交易日）"].dt.strftime("%Y-%m-%d")
gaps["空窗迄（下一交易日）"] = gaps["空窗迄（下一交易日）"].dt.strftime("%Y-%m-%d")
gaps = gaps.reset_index(drop=True)

print(f"間隔超過 {GAP_CALENDAR_MAX} 個日曆日的空窗共 {len(gaps)} 段（多數應為農曆年封關與連假）")
display(gaps)

間隔超過 5 個日曆日的空窗共 33 段（多數應為農曆年封關與連假）


,空窗起（前一交易日）,空窗迄（下一交易日）,日曆間隔天數,缺漏平日數
0,1999-02-10,1999-02-22,12,7
1,1999-09-20,1999-09-27,7,4
2,1999-12-28,2000-01-04,7,4
3,2000-02-01,2000-02-09,8,5
4,2001-01-18,2001-01-29,11,6
5,2002-02-06,2002-02-18,12,7
6,2003-01-28,2003-02-06,9,6
7,2004-01-16,2004-01-27,11,6
8,2005-02-03,2005-02-14,11,6
9,2006-01-25,2006-02-03,9,6


In [11]:
TABLES["資料空窗"] = gaps

# 這裡不自動判定對錯：長假與漏抓在數字上無法區分，需人工比對
long_gaps = gaps[gaps["缺漏平日數"] >= 6]
record("資料空窗（需人工確認）", True,
       f"共 {len(gaps)} 段空窗，最長缺漏 {int(gaps['缺漏平日數'].max())} 個平日；"
       f"其中缺漏 6 個平日以上者 {len(long_gaps)} 段，需人工比對是否對應農曆年封關或長連假")

===> [通過] 資料空窗（需人工確認）
      共 33 段空窗，最長缺漏 8 個平日；其中缺漏 6 個平日以上者 26 段，需人工比對是否對應農曆年封關或長連假


---
## 6. 極端變動

列出單日漲跌幅絕對值最大的前 15 天，並自動比對已知重大事件。

這一項的重點不是「有沒有大跌」—— 26 年跨越兩次腰斬，大跌本來就該存在。
重點是**每一個極端值都要對應得上真實事件**。
對不上的極端值，通常代表資料本身有錯（跳價、除權息處理錯誤、來源商的補值），
那種假的大跌會讓回測績效嚴重失真。

In [12]:
# (起, 迄, 事件名稱) —— 範圍窄的放前面，優先比對
KNOWN_EVENTS = [
    ("1999-09-20", "1999-10-15", "集集大地震"),
    ("2001-09-11", "2001-10-15", "美國 911 恐攻"),
    ("2004-03-19", "2004-04-09", "總統大選前槍擊事件"),
    ("2008-09-01", "2009-06-30", "金融海嘯"),
    ("2011-08-01", "2011-10-31", "歐債危機"),
    ("2015-08-01", "2015-09-30", "中國股災"),
    ("2018-02-01", "2018-02-28", "貿易戰（2 月）"),
    ("2018-10-01", "2018-12-31", "貿易戰（Q4）"),
    ("2020-02-15", "2020-04-30", "COVID-19"),
    ("2022-01-01", "2022-12-31", "升息循環"),
    ("2000-03-01", "2001-12-31", "網路泡沫破裂"),
]
KNOWN_EVENTS = [(pd.Timestamp(a), pd.Timestamp(b), n) for a, b, n in KNOWN_EVENTS]

def match_event(ts):
    for a, b, name in KNOWN_EVENTS:
        if a <= ts <= b:
            return name
    return "★ 無對應已知事件"

ret = df["Close"].pct_change()
extreme = (ret.abs().nlargest(TOP_N_EXTREME).index.sort_values())
ext = pd.DataFrame({
    "日期": extreme.strftime("%Y-%m-%d"),
    "報酬率": (ret.loc[extreme] * 100).round(2).values,
    "收盤價": df.loc[extreme, "Close"].round(2).values,
})
ext["對應事件"] = [match_event(t) for t in extreme]
ext["報酬率"] = ext["報酬率"].map("{:+.2f}%".format)
ext = ext.sort_values("日期").reset_index(drop=True)
display(ext)

,日期,報酬率,收盤價,對應事件
0,1999-02-08,+6.36%,"5,822.98",★ 無對應已知事件
1,1999-02-22,+8.89%,"6,313.63",★ 無對應已知事件
2,1999-07-16,-6.40%,"7,411.58",★ 無對應已知事件
3,2000-03-13,-6.55%,"8,811.95",網路泡沫破裂
4,2000-10-02,-6.35%,"6,024.07",網路泡沫破裂
5,2000-10-19,-6.46%,"5,081.28",網路泡沫破裂
6,2000-10-20,+6.37%,"5,404.78",網路泡沫破裂
7,2000-11-20,-9.46%,"4,845.21",網路泡沫破裂
8,2004-03-22,-6.68%,"6,359.92",總統大選前槍擊事件
9,2008-01-22,-6.51%,"7,581.96",★ 無對應已知事件


全期間收盤價走勢（對數座標），紅點為上表 15 個極端變動日。

用對數座標是因為指數從 5 千漲到 3 萬多，線性座標會讓早期的波動被壓扁到看不見。

In [13]:
fig, ax = plt.subplots()
ax.plot(df.index, df["Close"], lw=0.8, color="#3b6ea5", label="TWII Close")
ax.scatter(extreme, df.loc[extreme, "Close"], color="#d1495b", s=45, zorder=5,
           label=f"Top {TOP_N_EXTREME} daily moves")
ax.set_yscale("log")
ax.set_title("TAIEX (^TWII) Close, log scale — extreme daily moves marked")
ax.set_xlabel("Date")
ax.set_ylabel("Close (log)")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_17716\1062907680.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
TABLES["極端變動前 %d 名" % TOP_N_EXTREME] = ext

unmatched = ext[ext["對應事件"].str.contains("無對應")]
record(f"極端變動（前 {TOP_N_EXTREME} 名）",
       unmatched.empty,
       "全部可對應到已知重大事件"
       if unmatched.empty
       else "以下極端變動無法對應任何已知事件，需人工查證："
            + "、".join(f"{r['日期']} ({r['報酬率']})" for _, r in unmatched.iterrows()))

===> [異常] 極端變動（前 15 名）
      以下極端變動無法對應任何已知事件，需人工查證：1999-02-08 (+6.36%)、1999-02-22 (+8.89%)、1999-07-16 (-6.40%)、2008-01-22 (-6.51%)、2024-08-05 (-8.35%)、2025-04-07 (-9.70%)、2025-04-10 (+9.25%)


---
## 7. 價格合理性

四項結構性檢查，任何一項不通過都代表資料來源有問題：

1. `High < Low` —— 不可能發生
2. `Close` 或 `Open` 跑出 `High`–`Low` 區間 —— 不可能發生
3. 價格為 0 或負值 —— 指數不可能
4. 非休市日卻 `Volume == 0` —— 有交易日卻零成交量，通常代表成交量欄位是空的

第 4 項對本專案影響有限（策略不使用成交量），但仍要如實記錄，
因為它是判斷資料來源可靠度的線索。

In [15]:
price_cols = ["Open", "High", "Low", "Close"]

bad_hl    = df[df["High"] < df["Low"]]
bad_range = df[(df["Close"] > df["High"]) | (df["Close"] < df["Low"]) |
               (df["Open"]  > df["High"]) | (df["Open"]  < df["Low"])]
bad_price = df[(df[price_cols] <= 0).any(axis=1)]

price_check = pd.DataFrame({
    "檢查": ["High < Low",
             "Open/Close 超出 High–Low 區間",
             "價格 <= 0"],
    "違反筆數": [len(bad_hl), len(bad_range), len(bad_price)],
})
price_check["結果"] = np.where(price_check["違反筆數"] == 0, "通過", "★ 異常")
display(price_check)

for label, bad in [("High < Low", bad_hl),
                   ("Open/Close 超出區間", bad_range),
                   ("價格 <= 0", bad_price)]:
    if len(bad):
        print(f"--- {label} 明細（前 20 筆）---")
        display(bad.head(20))

,檢查,違反筆數,結果
0,High < Low,0,通過
1,Open/Close 超出 High–Low 區間,0,通過
2,價格 <= 0,0,通過


In [16]:
TABLES["價格合理性"] = price_check

n_bad = int(price_check["違反筆數"].sum())
record("價格合理性（OHLC 結構）", n_bad == 0,
       "High/Low/Open/Close 關係全部合理，無非正值價格"
       if n_bad == 0 else f"共 {n_bad} 筆違反，明細見上表")

===> [通過] 價格合理性（OHLC 結構）
      High/Low/Open/Close 關係全部合理，無非正值價格


In [17]:
zero_vol = df[df["Volume"] == 0]
ratio_zero = len(zero_vol) / len(df)

print(f"Volume == 0 的交易日：{len(zero_vol):,} 筆（{ratio_zero:.2%}）")

if len(zero_vol):
    zv_year = zero_vol.groupby(zero_vol.index.year).size().to_frame("零成交量日數")
    zv_year["該年交易日數"] = yearly["交易日數"]
    zv_year["占比"] = (zv_year["零成交量日數"] / zv_year["該年交易日數"]).map("{:.1%}".format)
    zv_year.index.name = "年份"
    display(zv_year)
    TABLES["零成交量日（依年份）"] = zv_year

record("成交量（非休市日 Volume == 0）", len(zero_vol) == 0,
       "無零成交量交易日"
       if len(zero_vol) == 0
       else f"{len(zero_vol):,} 個交易日 Volume 為 0（占 {ratio_zero:.2%}），"
            f"集中在 {zero_vol.index.min():%Y-%m-%d} ~ {zero_vol.index.max():%Y-%m-%d}；"
            f"本專案策略不使用成交量，但需人工確認來源")

Volume == 0 的交易日：987 筆（14.88%）


,零成交量日數,該年交易日數,占比
年份,,,
1999,241,241,100.0%
2000,245,245,100.0%
2001,245,245,100.0%
2002,248,248,100.0%
2019,5,241,2.1%
2022,1,246,0.4%
2024,1,242,0.4%
2025,1,243,0.4%


===> [異常] 成交量（非休市日 Volume == 0）
      987 個交易日 Volume 為 0（占 14.88%），集中在 1999-01-05 ~ 2025-01-21；本專案策略不使用成交量，但需人工確認來源


---
## 8. 開盤價可用性 ★ 本階段最關鍵的一項

後續策略**全部依賴「訊號成立後，隔日開盤成交」**。
如果資料來源的開盤價其實是用前一日收盤價填補的（不少免費資料源會這麼做），
那麼「隔日開盤買進」實際上等於「當日收盤買進」—— 回測會偷看到當天的資訊，績效整個失效。

判斷方法：統計 `Open == 前一日 Close` 的比例。
真實市場幾乎每天都有跳空，這個比例應該極低。**超過 5% 就標記為異常。**

同時看一下跳空幅度的分布 —— 如果開盤價是真的，跳空幅度應該是一個有寬度的分布，
而不是集中在 0。

In [18]:
prev_close = df["Close"].shift(1)
gap_open = (df["Open"] - prev_close) / prev_close

eq_exact = (df["Open"] == prev_close)                 # 完全相等
eq_tiny  = (gap_open.abs() < 1e-6)                    # 浮點誤差內視為相等

n_valid = int(prev_close.notna().sum())
ratio_exact = eq_exact.sum() / n_valid
ratio_tiny  = eq_tiny.sum() / n_valid

open_check = pd.DataFrame({
    "項目": ["Open 完全等於前一日 Close",
             "Open 與前一日 Close 差異 < 1e-6（浮點誤差內）",
             "可比對天數（扣除第一天）"],
    "筆數": [int(eq_exact.sum()), int(eq_tiny.sum()), n_valid],
    "比例": [f"{ratio_exact:.2%}", f"{ratio_tiny:.2%}", "—"],
})
display(open_check)

gap_stats = gap_open.describe().to_frame("跳空幅度")
gap_stats.loc["中位數絕對跳空"] = gap_open.abs().median()
display((gap_stats * 100).round(4).rename(columns={"跳空幅度": "跳空幅度 (%)"}))

,項目,筆數,比例
0,Open 完全等於前一日 Close,10,0.15%
1,Open 與前一日 Close 差異 < 1e-6（浮點誤差內）,10,0.15%
2,可比對天數（扣除第一天）,6631,—


,跳空幅度 (%)
count,"663,100.00"
mean,0.11
std,0.84
min,-6.67
25%,-0.18
50%,0.12
75%,0.45
max,7.21
中位數絕對跳空,0.34


In [19]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(gap_open.dropna() * 100, bins=200, range=(-5, 5), color="#3b6ea5")
ax.set_title("Overnight gap distribution: (Open - PrevClose) / PrevClose")
ax.set_xlabel("Gap (%)")
ax.set_ylabel("Trading days")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_17716\4077876501.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
TABLES["開盤價可用性"] = open_check

passed_open = ratio_tiny <= OPEN_EQ_PREV_CLOSE_MAX
record("開盤價可用性（Open vs 前一日 Close）", passed_open,
       f"Open 等於前一日 Close 的比例 {ratio_tiny:.2%}，"
       f"{'低於' if passed_open else '★ 超過'} {OPEN_EQ_PREV_CLOSE_MAX:.0%} 門檻；"
       f"中位數絕對跳空 {gap_open.abs().median() * 100:.3f}%，"
       f"{'開盤價可視為真實成交價，隔日開盤進場的假設成立' if passed_open else '開盤價疑似由前一日收盤填補，隔日開盤進場的回測假設不成立'}")

===> [通過] 開盤價可用性（Open vs 前一日 Close）
      Open 等於前一日 Close 的比例 0.15%，低於 5% 門檻；中位數絕對跳空 0.336%，開盤價可視為真實成交價，隔日開盤進場的假設成立


---
## 9. 檢查結果彙整與報告

把前面所有檢查記錄整理成一張總表，並輸出成 `data/data_quality_report.md`。

報告最後給出明確結論：這份資料能不能用於後續回測。
若有疑慮，具體指出是哪一項、影響什麼 —— **不自動修補任何問題，由人決定怎麼處理。**

In [21]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項檢查，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,檢查項目,結果,說明
0,欄位缺值,通過,所有欄位皆無缺值
1,每年交易日數,通過,1999–2025 年交易日數皆落在 230–255 之間
2,重複日期,通過,無重複日期
3,資料空窗（需人工確認）,通過,共 33 段空窗，最長缺漏 8 個平日；其中缺漏 6 個平日以上者 26 段，需人工比對是否...
4,極端變動（前 15 名）,異常,以下極端變動無法對應任何已知事件，需人工查證：1999-02-08 (+6.36%)、199...
5,價格合理性（OHLC 結構）,通過,High/Low/Open/Close 關係全部合理，無非正值價格
6,成交量（非休市日 Volume == 0）,異常,987 個交易日 Volume 為 0（占 14.88%），集中在 1999-01-05 ~...
7,開盤價可用性（Open vs 前一日 Close）,通過,Open 等於前一日 Close 的比例 0.15%，低於 5% 門檻；中位數絕對跳空 0....



共 8 項檢查，通過 6 項，異常 2 項


In [22]:
def to_md(dframe, index=True):
    """DataFrame 轉 markdown，tabulate 不在時退回純文字表格。"""
    try:
        return dframe.to_markdown(index=index)
    except Exception:
        return "```\n" + dframe.to_string(index=index) + "\n```"

# ---- 關鍵結論：哪些項目直接影響回測可用性 ----
BLOCKING = {"欄位缺值", "重複日期", "價格合理性（OHLC 結構）",
            "開盤價可用性（Open vs 前一日 Close）"}
failed = summary[summary["結果"] == "異常"]
blocking_failed    = failed[failed["檢查項目"].isin(BLOCKING)]
nonblocking_failed = failed[~failed["檢查項目"].isin(BLOCKING)]

L = []
A = L.append

A("# 台灣加權指數（^TWII）資料品質檢查報告")
A("")
A(f"- **標的**：`{TICKER}`")
A(f"- **下載期間**：{START} ~ {END}")
A(f"- **實際資料期間**：{df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}")
A(f"- **總筆數**：{len(df):,}")
A(f"- **資料來源**：yfinance（`auto_adjust=False`，未經任何清理）")
A(f"- **原始資料檔**：`{RAW_CSV}`")
A(f"- **報告產生時間**：{pd.Timestamp.now():%Y-%m-%d %H:%M:%S}")
A("")
A("> 樣本從 1999 年起算，比研究期間（2000-01-01 起）早一年，")
A("> 作為 MA50 的暖身資料，使 2000 年第一個交易日即可算出完整均線。")
A("")
A("---")
A("")
A("## 一、檢查結果總表")
A("")
A(to_md(summary, index=False))
A("")

A("## 二、各項明細")
A("")
for name, tbl in TABLES.items():
    A(f"### {name}")
    A("")
    A(to_md(tbl, index=not isinstance(tbl.index, pd.RangeIndex)))
    A("")

A("---")
A("")
A("## 三、結論")
A("")
if blocking_failed.empty:
    if nonblocking_failed.empty:
        A("### ✅ 資料可用於後續回測")
    else:
        A("### ✅ 資料可用於後續回測（有次要項目待確認，見下節）")
    A("")
    A("影響回測有效性的關鍵項目全部通過：")
    A("")
    for _, r in summary[summary["檢查項目"].isin(BLOCKING)].iterrows():
        A(f"- **{r['檢查項目']}**：{r['說明']}")
    A("")
    A(f"其中「開盤價可用性」最關鍵：Open 等於前一日 Close 的比例僅 {ratio_tiny:.2%}，"
      f"中位數絕對跳空 {gap_open.abs().median() * 100:.3f}%，"
      "代表開盤價是真實成交價而非前一日收盤填補，"
      "後續「訊號成立後隔日開盤成交」的回測假設成立。")
else:
    A("### ❌ 資料有關鍵問題，不建議直接用於回測")
    A("")
    A("以下項目直接影響回測有效性，必須先處理：")
    A("")
    for _, r in blocking_failed.iterrows():
        A(f"- **{r['檢查項目']}**：{r['說明']}")
A("")

if not nonblocking_failed.empty:
    A("### ⚠️ 需注意但不阻擋回測的項目")
    A("")
    for _, r in nonblocking_failed.iterrows():
        A(f"- **{r['檢查項目']}**：{r['說明']}")
    A("")

A("### 需人工確認的項目")
A("")
A(f"1. **資料空窗**（共 {len(gaps)} 段）：本檢查不自動判定對錯，因為「農曆年封關 / 長連假」"
  "與「資料漏抓」在數字上無法區分。請對照「資料空窗」明細表確認每段空窗都有對應假期。")
A(f"2. **極端變動**：前 {TOP_N_EXTREME} 大單日變動的事件對應是以日期區間自動比對，"
  "請人工複核對應關係是否合理。")
A("")
A("### 處理原則")
A("")
A("本 notebook **不自動修補任何資料問題**。上述所有發現僅作回報，")
A("是否調整、如何調整，由人判斷後於後續階段明確處理。")
A("")

report = "\n".join(L)
with open(REPORT_MD, "w", encoding="utf-8") as f:
    f.write(report)

print(f"報告已寫入：{REPORT_MD}（{len(report):,} 字元）")

報告已寫入：data\data_quality_report.md（11,749 字元）


---
## 階段 1 完成

產出檔案：

- `data/twii_raw.csv` —— 原始日線資料，未經清理
- `data/data_quality_report.md` —— 完整品質檢查報告

**下一階段（指標計算）請等待人工確認檢查報告後再進行。**

In [23]:
for p in [RAW_CSV, REPORT_MD]:
    print(f"{p:40s} {'存在' if os.path.exists(p) else '★ 不存在'}   "
          f"{os.path.getsize(p):>10,} bytes")

data\twii_raw.csv                        存在      537,740 bytes
data\data_quality_report.md              存在       14,430 bytes
